# Sakha GRPO Training Notebook

Train a hospital ward assistant using GRPO (Group Relative Policy Optimization) on the Sakha environment.

**Requirements**: GPU runtime (T4 is sufficient for 0.6B model)

**What this does**:
1. Installs Sakha environment and dependencies
2. Configures GRPO training with HF TRL
3. Runs training for N episodes
4. Plots reward curves and metrics

## 1. Setup - Install Dependencies

In [ ]:
import importlib.util, sys, os

# Idempotent setup: skip install if already available (e.g. after restart)
if importlib.util.find_spec("unsloth") is not None:
    print("Dependencies already installed. Skipping setup.")
    if os.path.exists("sakha"):
        %cd sakha
else:
    print("First run: installing dependencies...")
    !git clone -b main https://github.com/atharva-again/sakha.git
    %cd sakha
    !pip install uv -q
    !uv pip install --system -e ".[dev]"
    !uv pip install --system git+https://github.com/huggingface/transformers.git@main trl jmespath
    # Pin vllm==0.18.0 to avoid torch.compile crash in 0.19.x
    # See: https://github.com/unslothai/unsloth/issues/4841
    !uv pip install -qqq vllm==0.18.0 torchvision bitsandbytes xformers unsloth

    !uv pip install tqdm  # for interactive ux

First run: installing dependencies...
Cloning into 'sakha'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 420 (delta 10), reused 18 (delta 7), pack-reused 396 (from 1)
Receiving objects: 100% (420/420), 765.29 KiB | 2.93 MiB/s, done.
Resolving deltas: 100% (266/266), done.
/content/sakha
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 52.7 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 122 packages in 1.39s
Prepared 21 packages in 324ms
Installed 21 packages in 47ms
 + aiofile==3.9.0
 + caio==0.9.25
 + coverage==7.13.5
 + cyclopts==4.11.0
 + dnspython==2.8.0
 + email-validator==2.3.0
 + exceptiongroup==1.3.1
 + fastmcp==3.2.4
 + griffelib==2.0.2
 + jsonref==1.1.0
 + jsonschema-path==0.4.5
 + openapi-pydantic==0.5.1
 + openenv-core==0.2.3
 + pathable==0.5.0
 + py-key-value-aio==0.4.4
 + pytest-cov==7.1.0
 + rich-rst==1.3.2
 + sakha==0.1.0 (from file:///c

In [ ]:
# Optional: Mount Google Drive to cache HF models across sessions
# This saves ~5-10 min on every restart by reusing downloaded weights
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print(f"HF cache set to: {os.environ['HF_HOME']}")
except Exception:
    print("Google Drive not mounted. Models will cache in default ~/.cache/huggingface/")

Mounted at /content/drive
HF cache set to: /content/drive/MyDrive/hf_cache


## HF Token Setup

For Round 2 submission, use a T4 GPU (free Colab) for smoke tests and an A100 (HF credits) for full training.

Set your HuggingFace token in Colab secrets (left panel → 🔑) as  to access gated models.

## 2. Verify Environment

In [ ]:
import sys
sys.path.insert(0, "src")

import os
import torch
import random
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sakha.env import SakhaEnvironment
from sakha.models import SakhaAction, ActionType
from sakha.graders import score_easy_task, score_medium_task, score_hard_task
from sakha.grpo_training import (
    build_grpo_prompt,
    build_state_aligned_examples,
    parse_action_response_with_status,
    score_completion_action,
)
from tqdm import tqdm
import json
import gc

# Prevent vLLM from reconfiguring logging (crashes Jupyter/Colab's OutStream)
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

## 3. Configure Training

Adjust these parameters based on your GPU and time constraints:

In [ ]:

# Training configuration
MODEL = "Qwen/Qwen3-0.6B"          # Model to train (0.6B fits on T4)
TASK = "hard"                      # Task difficulty: easy | medium | hard
EPISODES = 100                       # Training examples (seed × state × policy). 200 ≈ 50 optimizer steps.
MAX_STEPS = 96                       # Max steps per episode (96 = full 8hr shift)
SEED = 42                            # Random seed for reproducibility

# Unsloth config (set USE_UNSLOTH=True for 4-bit training on T4)
USE_UNSLOTH = True                   # Use Unsloth for memory-efficient training
LOAD_IN_4BIT = True                  # 4-bit quantization (critical for T4)

# GRPO specific
NUM_GENERATIONS = 4                  # Responses per prompt
LEARNING_RATE = 1e-5                 # Learning rate
MAX_COMPLETION_LENGTH = 64           # Max tokens per completion (action call is ~5 tokens with /no_think)
MAX_SEQ_LENGTH = 2048                # Prompt + completion context for vLLM/Unsloth
EVAL_MAX_NEW_TOKENS = 32             # Plenty for one action call when thinking mode is OFF

# Checkpoint directory.
# To survive Colab session restarts, we put everything (LoRA checkpoints, eval caches,
# plot) in Google Drive when it's mounted. Override with SAKHA_OUTPUT_DIR if you want
# a different path. Falls back to ./grpo_output (ephemeral) only if no Drive is found.
def _default_output_dir() -> Path:
    env_dir = os.environ.get("SAKHA_OUTPUT_DIR")
    if env_dir:
        return Path(env_dir).expanduser()
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        return drive_root / "sakha_grpo"
    return Path("./grpo_output")


CHECKPOINT_DIR = _default_output_dir()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training config:")
print(f"  Model: {MODEL}")
print(f"  Task: {TASK}")
print(f"  Episodes: {EPISODES}")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Use Unsloth: {USE_UNSLOTH}")
print(f"  4-bit quant: {LOAD_IN_4BIT}")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")

def _completion_text(completion):
    if isinstance(completion, list):
        return str(completion[-1]["content"])
    return str(completion)


def _metadata_value(values, completion_idx: int, completion_count: int, default):
    """Select metadata whether TRL passes it per prompt or per generation."""
    if values is None:
        return default
    if not isinstance(values, (list, tuple)):
        return values
    if len(values) == completion_count:
        return values[completion_idx]
    if len(values) == 1:
        return values[0]
    prompt_idx = min(completion_idx // NUM_GENERATIONS, len(values) - 1)
    return values[prompt_idx]


def reward_func(prompts, completions, **kwargs):
    """State-aligned reward: parse the action, reconstruct prompt state, step once."""
    rewards = []
    completion_count = len(completions)
    for idx, completion in enumerate(completions):
        seed = int(_metadata_value(kwargs.get("seed"), idx, completion_count, SEED))
        replay_actions_json = _metadata_value(
            kwargs.get("replay_actions_json"), idx, completion_count, "[]"
        )
        rewards.append(
            score_completion_action(
                _completion_text(completion),
                task=TASK,
                seed=seed,
                replay_actions_json=str(replay_actions_json),
            )
        )
    return rewards

Training config:
  Model: Qwen/Qwen3-0.6B
  Task: hard
  Episodes: 200
  Max steps: 96
  Use Unsloth: True
  4-bit quant: True


## 5. Create Dataset and Configure GRPO

In [ ]:
def create_prompt(task: str, episode_id: int = 0):
    env = SakhaEnvironment(task=task)
    obs = env.reset(seed=SEED + episode_id)
    return build_grpo_prompt(obs, task=task, episode_id=episode_id)

# Build dataset
training_examples = build_state_aligned_examples(
    task=TASK,
    episodes=EPISODES,
    seed=SEED,
    max_steps=MAX_STEPS,
)
dataset = Dataset.from_dict(training_examples)

print(f"Dataset size: {len(dataset)} prompts")

Dataset size: 200 prompts


## 6. Pre-Training Evaluation — Base LLM

In [ ]:
# RUN THIS FIRST to avoid Unsloth monkey-patch conflict

SYSTEM_PROMPT = (
    "You are a hospital ward assistant managing patients. "
    "Available actions: review_patient(patient_id), check_vitals(patient_id), "
    "administer_medicine(patient_id), alert_doctor(patient_id), escalate(patient_id), "
    "update_chart(patient_id), prepare_discharge(patient_id), medication_round(), "
    "ward_sweep(), noop(). "
    "Return exactly one final action call. Do not explain."
)

def build_eval_prompt(obs):
    pending = obs.ward_state.pending_tasks[:5] if obs.ward_state.pending_tasks else []
    tasks_str = ", ".join(f"{t.required_action}({t.patient_id})" for t in pending) or "No pending tasks"
    return (
        f"Step {obs.ward_state.current_step}. "
        f"Patients: {len(obs.ward_state.patients)}. "
        f"Pending: {obs.pending_count}. "
        f"Tasks: {tasks_str}. "
        f"What action do you take?"
    )

def parse_action_response(response):
    action, _ = parse_action_response_with_status(response)
    return action

def save_eval_cache(results, cache_path):
    """Save evaluation results to JSON for resume across sessions."""
    with open(cache_path, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"💾 Eval results saved to {cache_path}")

def load_eval_cache(cache_path):
    """Load cached evaluation results if available."""
    if cache_path.exists():
        with open(cache_path, "r") as f:
            results = json.load(f)
        print(f"⚡ Loaded cached eval results from {cache_path}")
        return results
    return None

def run_llm_eval_batched(task, model, tokenizer, seeds, max_steps):
    """Vectorized evaluation: runs all seeds in parallel per step for ~5x speedup."""
    from sakha.graders import score_easy_task, score_medium_task, score_hard_task
    device = next(model.parameters()).device
    print(f"Running batched eval on device: {device} | {len(seeds)} seeds × {max_steps} steps")

    TASK_GRADERS = {"easy": score_easy_task, "medium": score_medium_task, "hard": score_hard_task}
    PATIENT_COUNTS = {"easy": 5, "medium": 8, "hard": 18}
    pc = PATIENT_COUNTS[task]
    grader = TASK_GRADERS[task]

    # Initialize all environments at once
    envs = []
    observations = []
    trajectories = []
    step_rewards_all = []
    parse_failures = []
    for seed in seeds:
        env = SakhaEnvironment(patient_count=pc, task=task)
        obs = env.reset(seed=seed)
        envs.append(env)
        observations.append(obs)
        trajectories.append([obs])
        step_rewards_all.append([])
        parse_failures.append(0)


    active_mask = [True] * len(seeds)
    pbar = tqdm(total=max_steps, desc=f"Eval {task} (batched, {len(seeds)} seeds)")

    for step in range(max_steps):
        # Collect indices of environments that are still running
        active_indices = [i for i, active in enumerate(active_mask) if active]
        if not active_indices:
            pbar.update(max_steps - step)
            break

        # Build prompts using the SAME builder as training so the model sees the
        # exact distribution it was optimized on. Mismatched train/eval prompts
        # were the main cause of the trained-vs-base regression.
        # NOTE: Qwen3 has thinking mode ON by default — without enable_thinking=False
        # the model emits <think>...</think> reasoning before every action, turning
        # a 5-token action call into 100+ tokens and making eval ~5-8x slower.
        prompt_texts = []
        for i in active_indices:
            messages = build_grpo_prompt(
                observations[i], task=task, episode_id=seeds[i]
            )
            try:
                prompt_text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                # Older tokenizers / non-Qwen templates don't accept the kwarg.
                prompt_text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            prompt_texts.append(prompt_text)

        # Batch tokenize with left-padding
        inputs = tokenizer(
            prompt_texts, return_tensors="pt", padding=True
        ).to(device)

        # Single batched generation call
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=EVAL_MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Parse responses and step environments
        input_lens = inputs["attention_mask"].sum(dim=1)
        newly_done = []
        for batch_idx, env_idx in enumerate(active_indices):
            input_len = input_lens[batch_idx].item()
            response = tokenizer.decode(
                outputs[batch_idx][input_len:], skip_special_tokens=True
            )
            action, parsed_ok = parse_action_response_with_status(response)
            if not parsed_ok:
                parse_failures[env_idx] += 1
            obs = envs[env_idx].step(action)
            observations[env_idx] = obs
            trajectories[env_idx].append(obs)
            step_rewards_all[env_idx].append(obs.reward)

            if obs.done:
                active_mask[env_idx] = False
                newly_done.append(env_idx)


        # Log
        active_count = sum(active_mask)
        sample_idx = active_indices[0]
        sample_action = parse_action_response(
            tokenizer.decode(outputs[0][input_lens[0].item():], skip_special_tokens=True)
        )
        pbar.set_postfix({
            "active": f"{active_count}/{len(seeds)}",
            "action": sample_action.action_type.name,
        })
        for idx in newly_done:
            pbar.write(f"  🏁 Seed {seeds[idx]} finished at step {step}")
        pbar.update(1)

    pbar.close()

    # Compute per-episode results
    episodes = []
    for i, seed in enumerate(seeds):
        grader_score = grader(trajectories[i])
        metrics = envs[i].episode_metrics
        ep = {
            "seed": seed, "grader_score": grader_score,
            "total_reward": sum(step_rewards_all[i]),
            "critical_incidents_resolved": metrics.critical_incidents_resolved,
            "critical_incidents_missed": metrics.critical_incidents_missed,
            "overdue_tasks": metrics.overdue_tasks,
            "noop_steps": metrics.noop_steps,
            "parse_failures": parse_failures[i],
            "discharges_prepared": metrics.discharges_prepared,
        }
        episodes.append(ep)
        print(
            f"  Seed {seed:3}: Reward {ep['total_reward']:6.2f} | "
            f"Grader {grader_score:.4f} | Steps {len(trajectories[i])-1} | "
            f"Parse failures {parse_failures[i]}"
        )

    def mean(key):
        return sum(e[key] for e in episodes) / len(episodes)
    return {
        "task": task, "policy": "base_llm", "episodes": episodes,
        "summary": {
            "mean_grader_score": mean("grader_score"),
            "mean_total_reward": mean("total_reward"),
            "mean_critical_incidents_resolved": mean("critical_incidents_resolved"),
            "mean_critical_incidents_missed": mean("critical_incidents_missed"),
            "mean_overdue_tasks": mean("overdue_tasks"),
            "mean_noop_steps": mean("noop_steps"),
            "mean_discharges_prepared": mean("discharges_prepared"),
        }
    }

# --- Base Eval: Check cache first, then run if needed ---
BASE_CACHE = CHECKPOINT_DIR / "base_eval_cache.json"
base_results = load_eval_cache(BASE_CACHE)

if base_results is None:
    print("Loading base Qwen3-0.6B for zero-shot evaluation...")
    base_tokenizer = AutoTokenizer.from_pretrained(
        MODEL, trust_remote_code=True, padding_side="left"
    )
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model.eval()

    EVAL_SEEDS = list(range(42, 52))
    base_results = run_llm_eval_batched(TASK, base_model, base_tokenizer, EVAL_SEEDS, MAX_STEPS)
    save_eval_cache(base_results, BASE_CACHE)

    # Free VRAM for training
    del base_model
    del base_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
else:
    EVAL_SEEDS = list(range(42, 52))

print(f"Base LLM mean grader score: {base_results['summary']['mean_grader_score']:.4f}")

Loading base Qwen3-0.6B for zero-shot evaluation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


Running batched eval on device: cuda:0 | 10 seeds × 96 steps


Eval hard (batched, 10 seeds): 100%|██████████| 96/96 [22:59<00:00, 14.37s/it, active=0/10, action=CHECK_VITALS]


  🏁 Seed 42 finished at step 95
  🏁 Seed 43 finished at step 95
  🏁 Seed 44 finished at step 95
  🏁 Seed 45 finished at step 95
  🏁 Seed 46 finished at step 95
  🏁 Seed 47 finished at step 95
  🏁 Seed 48 finished at step 95
  🏁 Seed 49 finished at step 95
  🏁 Seed 50 finished at step 95
  🏁 Seed 51 finished at step 95
  Seed  42: Reward   1.54 | Grader 0.2986 | Steps 96 | Parse failures 16
  Seed  43: Reward  -0.39 | Grader 0.2886 | Steps 96 | Parse failures 11
  Seed  44: Reward  -2.27 | Grader 0.1574 | Steps 96 | Parse failures 19
  Seed  45: Reward  -2.04 | Grader 0.0000 | Steps 96 | Parse failures 43
  Seed  46: Reward   0.65 | Grader 0.3280 | Steps 96 | Parse failures 12
  Seed  47: Reward  -1.63 | Grader 0.0000 | Steps 96 | Parse failures 35
  Seed  48: Reward  -0.07 | Grader 0.2778 | Steps 96 | Parse failures 14
  Seed  49: Reward  -0.75 | Grader 0.2563 | Steps 96 | Parse failures 17
  Seed  50: Reward  -0.99 | Grader 0.2230 | Steps 96 | Parse failures 15
  Seed  51: Reward  -2.

Save the base model in the drive

# 7. Load Model with Unsloth (Optional)

In [ ]:
# Decide up front whether we'll need to actually train. If a usable checkpoint
# already exists in CHECKPOINT_DIR we skip the (expensive) Unsloth/vLLM load too,
# so a session restart doesn't redo the work and VRAM stays free for trained-eval.
#   SAKHA_FORCE_RETRAIN=1   -> train fresh
#   SAKHA_RESUME_TRAINING=1 -> continue from latest checkpoint
_pretraining_ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
_force_retrain = os.environ.get("SAKHA_FORCE_RETRAIN") == "1"
_resume_training = os.environ.get("SAKHA_RESUME_TRAINING") == "1"
_will_train = (not _pretraining_ckpts) or _force_retrain or _resume_training

if _will_train and USE_UNSLOTH:
    from unsloth import FastLanguageModel, PatchFastRL
    PatchFastRL("GRPO", FastLanguageModel)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        fast_inference=True,
        gpu_memory_utilization=0.6,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    print(f"Unsloth model loaded for training: {MODEL}")
elif _will_train:
    model = MODEL
    tokenizer = None
else:
    print(
        f"Found {len(_pretraining_ckpts)} existing checkpoint(s) in {CHECKPOINT_DIR}. "
        "Skipping Unsloth load (will resume to trained-eval)."
    )
    model = None
    tokenizer = None

/tmp/ipykernel_1135/1051103320.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.


we do not yet support fast inference for unsloth/qwen3-0.6b-unsloth-bnb-4bit


==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: vLLM loading unsloth/qwen3-0.6b-bnb-4bit with actual GPU utilization = 69.18%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 48.
Unsloth: vLLM's KV Cache can use up to 9.36 GB. Also swap space = 0 GB.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_inductor` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `swap_space` is not supported in vLLM. Skipping.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.


...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-0.6b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth model loaded for training: Qwen/Qwen3-0.6B


# 8. Configure GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    num_train_epochs=1,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    report_to=[],
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=False,
    seed=SEED,
    remove_unused_columns=False,
    use_vllm=USE_UNSLOTH,
    vllm_gpu_memory_utilization=0.3 if USE_UNSLOTH else None,
)

# 9. Trainer Setup & Training

In [ ]:
trainer_kwargs = {
    "train_dataset": dataset,
    "reward_funcs": reward_func,
    "args": grpo_config,
}

if USE_UNSLOTH:
    trainer_kwargs["model"] = model
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["model"] = MODEL

# --- Training control flow ---
# Default: if checkpoints already exist in CHECKPOINT_DIR, skip training entirely
# and jump to the trained-eval cell below. Override with the env vars described
# in the Unsloth-load cell (SAKHA_FORCE_RETRAIN / SAKHA_RESUME_TRAINING).
if not _will_train:
    print(
        f"Skipping training — using existing checkpoints in {CHECKPOINT_DIR}. "
        "Set SAKHA_FORCE_RETRAIN=1 or SAKHA_RESUME_TRAINING=1 to override."
    )
else:
    trainer = GRPOTrainer(**trainer_kwargs)
    resume_path = (
        str(_pretraining_ckpts[-1])
        if (_pretraining_ckpts and _resume_training)
        else None
    )
    if resume_path:
        print(f"Resuming training from {resume_path}")
    else:
        print(f"Starting training: {EPISODES} episodes, task={TASK}")
    trainer.train(resume_from_checkpoint=resume_path)

Starting training: 200 episodes, task=hard


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.167900,-0.380000,0.141421,213.500000,101.000000,320.000000,0.250000,178.000000,101.000000,283.000000,0.000022,-0.380000,0.141421
2,-0.034600,2.395000,1.250000,192.500000,127.000000,252.000000,0.000000,192.500000,127.000000,252.000000,0.000020,2.395000,1.250000
3,-0.203400,-0.230000,0.100000,224.250000,147.000000,349.000000,0.000000,224.250000,147.000000,349.000000,0.000024,-0.230000,0.100000
4,-0.234800,-0.040000,0.184752,289.750000,195.000000,368.000000,0.500000,211.500000,195.000000,228.000000,0.000022,-0.040000,0.184752
5,-0.485400,-0.040000,0.184752,109.000000,11.000000,201.000000,0.000000,109.000000,11.000000,201.000000,0.000028,-0.040000,0.184752
6,-0.000100,-0.495000,0.590000,193.000000,193.000000,193.000000,0.750000,193.000000,193.000000,193.000000,0.000028,-0.495000,0.590000
7,0.000000,0.220000,0.000000,179.500000,137.000000,220.000000,0.000000,179.500000,137.000000,220.000000,0.000026,0.220000,0.000000
8,-0.028400,-0.170000,0.060000,198.000000,170.000000,215.000000,0.000000,198.000000,170.000000,215.000000,0.000028,-0.170000,0.060000
9,0.000000,0.520000,0.000000,151.250000,91.000000,193.000000,0.000000,151.250000,91.000000,193.000000,0.000034,0.520000,0.000000
10,-0.007300,-0.180000,0.200000,273.500000,208.000000,336.000000,0.000000,273.500000,208.000000,336.000000,0.000057,-0.180000,0.200000


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.167900,-0.380000,0.141421,213.500000,101.000000,320.000000,0.250000,178.000000,101.000000,283.000000,0.000022,-0.380000,0.141421
2,-0.034600,2.395000,1.250000,192.500000,127.000000,252.000000,0.000000,192.500000,127.000000,252.000000,0.000020,2.395000,1.250000
3,-0.203400,-0.230000,0.100000,224.250000,147.000000,349.000000,0.000000,224.250000,147.000000,349.000000,0.000024,-0.230000,0.100000
4,-0.234800,-0.040000,0.184752,289.750000,195.000000,368.000000,0.500000,211.500000,195.000000,228.000000,0.000022,-0.040000,0.184752
5,-0.485400,-0.040000,0.184752,109.000000,11.000000,201.000000,0.000000,109.000000,11.000000,201.000000,0.000028,-0.040000,0.184752
6,-0.000100,-0.495000,0.590000,193.000000,193.000000,193.000000,0.750000,193.000000,193.000000,193.000000,0.000028,-0.495000,0.590000
7,0.000000,0.220000,0.000000,179.500000,137.000000,220.000000,0.000000,179.500000,137.000000,220.000000,0.000026,0.220000,0.000000
8,-0.028400,-0.170000,0.060000,198.000000,170.000000,215.000000,0.000000,198.000000,170.000000,215.000000,0.000028,-0.170000,0.060000
9,0.000000,0.520000,0.000000,151.250000,91.000000,193.000000,0.000000,151.250000,91.000000,193.000000,0.000034,0.520000,0.000000
10,-0.007300,-0.180000,0.200000,273.500000,208.000000,336.000000,0.000000,273.500000,208.000000,336.000000,0.000057,-0.180000,0.200000


# 10. Post-Training Evaluation — Trained LLM

In [ ]:
TRAINED_CACHE = CHECKPOINT_DIR / "trained_eval_cache.json"
trained_results = load_eval_cache(TRAINED_CACHE)

if trained_results is None:
    ckpts = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
    if not ckpts:
        print("No checkpoints found. Skipping trained eval.")
        trained_results = None
    else:
        CHECKPOINT_PATH = str(ckpts[-1])
        print(f"Loading trained checkpoint: {CHECKPOINT_PATH}")

        # Free training-time resources before loading the eval model so we don't
        # OOM (Unsloth+vLLM hold a lot of VRAM after train()).
        try:
            del trainer
        except NameError:
            pass
        try:
            del model
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()

        # Always evaluate via plain transformers + PEFT so the trained-vs-base
        # comparison is apples-to-apples (same inference framework as base eval)
        # and we don't surface the misleading "lm_head.weight ... newly
        # initialized" warning that the Unsloth/vLLM path emits for tied-
        # embedding models like Qwen3.
        from peft import PeftModel
        trained_tokenizer = AutoTokenizer.from_pretrained(
            MODEL, trust_remote_code=True, padding_side="left"
        )
        base_for_trained = AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        trained_model = PeftModel.from_pretrained(base_for_trained, CHECKPOINT_PATH)
        trained_model = trained_model.merge_and_unload()
        trained_model.eval()

        if trained_tokenizer.pad_token is None:
            trained_tokenizer.pad_token = trained_tokenizer.eos_token

        print("Running evaluation on trained model...")
        trained_results = run_llm_eval_batched(TASK, trained_model, trained_tokenizer, EVAL_SEEDS, MAX_STEPS)
        trained_results["policy"] = "trained_llm"
        save_eval_cache(trained_results, TRAINED_CACHE)
        print(f"Trained mean grader score: {trained_results['summary']['mean_grader_score']:.4f}")

        del trained_model
        del base_for_trained
        del trained_tokenizer
        gc.collect()
        torch.cuda.empty_cache()


Loading trained checkpoint: grpo_output/checkpoint-200
INFO 04-25 21:42:03 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen3-0.6b-bnb-4bit with actual GPU utilization = 59.3%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 32.
Unsloth: vLLM's KV Cache can use up to 7.93 GB. Also swap space = 0 GB.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationCo

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['k_norm', 'cross_attn_input_layernorm', 'q_norm', 'post_layernorm', 'norm2', 'ffn_norm', 'input_layernorm', 'norm', 'attention_norm', 'layer_norm2', 'norm1', 'post_feedforward_layernorm', 'pre_feedforward_layernorm', 'cross_attn_post_attention_layernorm', 'layer_norm1', 'post_attention_layernorm']


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Running evaluation on trained model...
Running batched eval on device: cuda:0 | 10 seeds × 96 steps


Eval hard (batched, 10 seeds): 100%|██████████| 96/96 [10:07<00:00,  6.32s/it, active=0/10, action=NOOP]

  🏁 Seed 42 finished at step 95
  🏁 Seed 43 finished at step 95
  🏁 Seed 44 finished at step 95
  🏁 Seed 45 finished at step 95
  🏁 Seed 46 finished at step 95
  🏁 Seed 47 finished at step 95
  🏁 Seed 48 finished at step 95
  🏁 Seed 49 finished at step 95
  🏁 Seed 50 finished at step 95
  🏁 Seed 51 finished at step 95
  Seed  42: Reward  -2.86 | Grader 0.0000 | Steps 96
  Seed  43: Reward  -0.73 | Grader 0.2320 | Steps 96
  Seed  44: Reward  -0.69 | Grader 0.2359 | Steps 96
  Seed  45: Reward  -2.32 | Grader 0.0000 | Steps 96
  Seed  46: Reward  -0.29 | Grader 0.2453 | Steps 96
  Seed  47: Reward  -1.07 | Grader 0.2287 | Steps 96
  Seed  48: Reward  -1.32 | Grader 0.1215 | Steps 96
  Seed  49: Reward  -0.95 | Grader 0.2360 | Steps 96
  Seed  50: Reward  -2.46 | Grader 0.0000 | Steps 96
  Seed  51: Reward  -1.53 | Grader 0.0000 | Steps 96
💾 Eval results saved to grpo_output/trained_eval_cache.json
Trained mean grader score: 0.1299


## 11. Base vs Trained Comparison

Side-by-side metrics and bar chart.

In [ ]:
if trained_results is not None:
    b = base_results["summary"]
    t = trained_results["summary"]

    print("\n## Base LLM vs Trained LLM Comparison\n")
    print("| Metric | Base LLM | Trained | Delta |")
    print("|--------|----------|---------|-------|")
    print(f"| Grader Score | {b['mean_grader_score']:.4f} | {t['mean_grader_score']:.4f} | {t['mean_grader_score'] - b['mean_grader_score']:+.4f} |")
    print(f"| Total Reward | {b['mean_total_reward']:.4f} | {t['mean_total_reward']:.4f} | {t['mean_total_reward'] - b['mean_total_reward']:+.4f} |")
    print(f"| Critical Resolved | {b['mean_critical_incidents_resolved']:.2f} | {t['mean_critical_incidents_resolved']:.2f} | {t['mean_critical_incidents_resolved'] - b['mean_critical_incidents_resolved']:+.2f} |")
    print(f"| Critical Missed | {b['mean_critical_incidents_missed']:.2f} | {t['mean_critical_incidents_missed']:.2f} | {t['mean_critical_incidents_missed'] - b['mean_critical_incidents_missed']:+.2f} |")
    print(f"| Overdue Tasks | {b['mean_overdue_tasks']:.2f} | {t['mean_overdue_tasks']:.2f} | {t['mean_overdue_tasks'] - b['mean_overdue_tasks']:+.2f} |")
    print(f"| NOOP Steps | {b['mean_noop_steps']:.2f} | {t['mean_noop_steps']:.2f} | {t['mean_noop_steps'] - b['mean_noop_steps']:+.2f} |")

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    metrics = ["Grader Score", "Total Reward", "Critical Resolved", "Overdue Tasks"]
    base_vals = [b['mean_grader_score'], b['mean_total_reward'], b['mean_critical_incidents_resolved'], b['mean_overdue_tasks']]
    trained_vals = [t['mean_grader_score'], t['mean_total_reward'], t['mean_critical_incidents_resolved'], t['mean_overdue_tasks']]
    x = range(len(metrics))
    width = 0.35
    ax.bar([i - width/2 for i in x], base_vals, width, label='Base LLM', color='#e74c3c')
    ax.bar([i + width/2 for i in x], trained_vals, width, label='Trained LLM', color='#2ecc71')
    ax.set_ylabel('Value')
    ax.set_title('Base vs Trained LLM Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = CHECKPOINT_DIR / "base_vs_trained.png"
    plt.savefig(plot_path, dpi=150)
    print(f"Plot saved to {plot_path}")
else:
    print("Skipping comparison — no trained eval results available.")

NameError: name 'trained_results' is not defined

## 13. Download Results

In [ ]:
# Zip results for download. CHECKPOINT_DIR may live on Drive
# (e.g. /content/drive/MyDrive/sakha_grpo) — zip it from wherever it is.
import shutil
zip_base = "grpo_results"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(CHECKPOINT_DIR))
print(f"Archived {CHECKPOINT_DIR} → {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("(Skip download — not in Colab)")

---

## Summary

This notebook demonstrated GRPO training on the Sakha hospital ward environment.

**Key takeaways**:
- Sakha provides a realistic hospital workflow simulation with 96-step episodes
- GRPO with Unsloth 4-bit training fits on a free Colab T4 GPU
- Compare base LLM vs trained LLM on identical seeds to measure RL improvement
- Small models (0.6B) can learn basic patterns, but larger models (4B+) show better reasoning

**Next steps**:
- Try larger models (Qwen3-4B) for better performance
- Increase episodes to 500+ for meaningful convergence
- Add eval split to track generalization
- Experiment with different reward shaping